# TFIM Gibbs states: SVM balanced accuracies

Runs the HS- and IPR-kernel SVMs for $L = 20, 50, 100$ and $\beta = 0.1, \dots, 2$. Writes Table V to `figures/tables/thermal_svm.json` and caches the full sweep, including the split-averaged decision functions, in `data/processed/thermal_svm_results.pkl`, which `figures/make_figures.py` reads for Figs. 6 and 11.

Requires `data/processed/tfim_thermal/` from `process_data.ipynb`. With 100 splits the sweep takes about three hours on a laptop.

In [ ]:
import pickle

import matplotlib.pyplot as plt

from ipr_kernel.config import PROCESSED_DIR
from ipr_kernel.tables import TABLES_DIR, svm_rows, write_table
from ipr_kernel.thermal_svm import run_thermal_kernel_svm_beta_sweep

In [ ]:
# Settings of the manuscript.
L_VALUES = [20, 50, 100]
N_SPLITS = 100
CACHE = PROCESSED_DIR / "thermal_svm_results.pkl"

In [ ]:
results = {}
for L in L_VALUES:
    fig, _, results[L] = run_thermal_kernel_svm_beta_sweep(L=L, kinds=("ipr", "hs"), source="processed",
                                                          test_frac=0.10, local_delta=0.25, n_splits=N_SPLITS,
                                                          rng_seed=12345, C=1, show_plot_title=False)
    plt.close(fig)
with open(CACHE, "wb") as f:
    pickle.dump(results, f)

In [ ]:
tables = {str(L): {kind: svm_rows([round(float(b), 1) for b in r["betas"]], r["by_kind"][kind])
                   for kind in ("ipr", "hs")} for L, r in results.items()}
write_table(TABLES_DIR / "thermal_svm.json", tables,
            f"Balanced accuracies of tab:svm_thermal (periodic HS kernel; {N_SPLITS} splits, test fraction 0.1, "
            "local delta = 0.25). Columns: beta, global mean, global std, local mean, local std.")